In [ ]:
import MeshFEM, mesh
import discrete_shell
import numpy as np
import tri_mesh_viewer
import loads
import benchmark

from matplotlib import pyplot as plt

In [ ]:
m = mesh.Mesh('../3rdparty/MeshFEM/misc/examples/meshes/lilium.msh', embeddingDimension=3)
ds = discrete_shell.DiscreteShell(m, youngModulus=100)

In [ ]:
v = tri_mesh_viewer.Viewer(ds, wireframe=True)
v.show()

In [ ]:
v.update()

In [ ]:
attachmentPoints = [loads.AttachmentPointCoordinate([i], [1]) for i in range(ds.numVars())]
targets = [loads.AttachmentPointCoordinate(v) for v in m.vertices().ravel()]
s=discrete_shell.Springs(ds, attachmentPoints, targets, 1e2)

In [ ]:
import sim_utils
# Keep the mesh boundary vertices on the ground
fixedVars = sim_utils.getBBoxVars(ds, sim_utils.BBoxFace.MIN_Z, tol=1e-2, displacementComponents=[2])

In [ ]:
import py_newton_optimizer
nopts = py_newton_optimizer.NewtonOptimizerOptions()
nopts.niter = 1000
nopts.verbose = 1
nopts.gradTol = 1e-6
attachmentPoints = [loads.AttachmentPointCoordinate([i], [1]) for i in range(ds.numVars())]
targets = [loads.AttachmentPointCoordinate(v) for v in m.vertices().ravel()]
s=discrete_shell.Springs(ds, attachmentPoints, targets, 1e2)

In [ ]:
ds.delta = 0.01

In [ ]:
benchmark.reset()
ds.computeEquilibrium(opts=nopts, cb=tri_mesh_viewer.ViewUpdater(v), loads=[s], fixedVars=fixedVars)
v.update()
benchmark.report()

In [ ]:
ds.setVars(ds.getVars() + 1e-3 * np.random.normal(size=ds.numVars()))

import fd_validation
fd_validation.gradConvergencePlot(ds)

In [ ]:
fd_validation.hessConvergencePlot(ds)